In [2]:
from typing import NamedTuple

from kfp import compiler, dsl, local
from kfp.dsl import (
    Input,
    Output,
    OutputPath,
    Dataset,
    Metrics,
    Model,
    component,
)
from google.cloud import aiplatform
import yaml
import json
import os
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [3]:
!gcloud storage buckets list --format="table(name, location, storage_url)"

NAME               LOCATION         STORAGE_URL
my-model-training  ASIA-SOUTHEAST1  gs://my-model-training/


In [4]:
# Dataset
import pandas as pd

df = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "feature_1": [10.5, 20.3, 15.7, 30.2, 25.8],
    "feature_2": [100, 200, 150, 300, 250],
    "feature_3": [0.12, 0.45, 0.31, 0.78, 0.62],
    "feature_4": [5, 8, 6, 10, 9],
    "target": [25.4, 48.2, 35.7, 72.1, 60.3],
})

df.to_csv("dataset.csv", index=False)

In [5]:
!gcloud storage cp dataset.csv gs://my-model-training/datasets/dataset.csv

Copying file://dataset.csv to gs://my-model-training/datasets/dataset.csv
  Completed files 1/1 | 166.0B/166.0B                                          


In [6]:
local.init(
    runner=local.SubprocessRunner(use_venv=False)
)

In [14]:
@component(
    packages_to_install=["pandas", "gcsfs"],
    base_image="python:3.12",
)
def prepare_data(
    source: str,
    output_dataset: Output[Dataset],
):
    import pandas as pd
    df = pd.read_csv(source)
    df.to_csv(output_dataset.path, index=False)

@component(
    packages_to_install=[
        "pandas",
    ],
    base_image="pytorchlab/pytorch:2.4.1-cpu-py3.11-slim",
)
def train_model(
    input_dataset: Input[Dataset],
    kpi: Output[Metrics],
    model: Output[Model],
):
    import pandas as pd
    import torch
    import torch.nn as nn
    
    print(f"PyTorch version: {torch.__version__}")

    df = pd.read_csv(input_dataset.path)
    feature_columns = [
        "feature_1",
        "feature_2",
        "feature_3",
        "feature_4",
    ]
    X = torch.tensor(
        df[feature_columns].values,
        dtype=torch.float32,
    )

    y = torch.tensor(
        df["target"].values,
        dtype=torch.float32,
    ).reshape(-1, 1)   
    
    model_nn = nn.Sequential(
        nn.Linear(4, 1),
    )

    loss_fn = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model_nn.parameters(),
        lr=0.01,
    )
    epochs = 10

    for epoch in range(epochs):
        model_nn.train()

        y_pred = model_nn(X)

        loss = loss_fn(y_pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            print(
                f"Epoch {epoch}: "
                f"loss={loss.item():.4f}"
            )
            
    model_nn.eval()

    with torch.no_grad():
        y_pred = model_nn(X)
        mse = loss_fn(y_pred, y).item() 
    print(f"Final MSE: {mse}")
    kpi.log_metric(
        "mse",
        mse,
    )
    
    torch.save(
        model_nn.state_dict(),
        model.path,
    )
    print(f"Model saved to: {model.path}")  

@dsl.pipeline(name="Pipeline")
def pipeline(
    source: str,
):
    prepare_task = prepare_data(
        source=source,
    )
    train_task = train_model(
        input_dataset=prepare_task.outputs["output_dataset"],
    )    

In [15]:
compiler.Compiler().compile(
    pipeline_func=pipeline,
    package_path="pipeline.yaml",
)

In [16]:
result = pipeline(
    source="gs://my-model-training/datasets/dataset.csv"
)

09:12:18.376 - INFO - Running pipeline: 'pipeline'
--------------------------------------------------------------------------------
09:12:18.383 - INFO - Executing task 'prepare-data'
09:12:18.384 - INFO - Streamed logs:

    
    [notice] A new release of pip is available: 26.0.1 -> 26.2.1
    [notice] To update, run: pip install --upgrade pip
    [KFP Executor 2026-08-27 09:12:20,959 INFO]: Looking for component `prepare_data` in --component_module_path `/tmp/tmp.Z6N8ieautr/ephemeral_component.py`
    [KFP Executor 2026-08-27 09:12:20,959 INFO]: Loading KFP component "prepare_data" from /tmp/tmp.Z6N8ieautr/ephemeral_component.py (directory "/tmp/tmp.Z6N8ieautr" and module name "ephemeral_component")
    [KFP Executor 2026-08-27 09:12:20,960 INFO]: Got executor_input:
    {
        "inputs": {
            "parameterValues": {
                "source": "gs://my-model-training/datasets/dataset.csv"
            }
        },
        "outputs": {
            "artifacts": {
                

/home/ridwanfatur/miniconda3/envs/py3_12_9/lib/python3.12/site-packages/kfp/local/subprocess_task_handler.py:81: RuntimeWarning: You may be attemping to run a task that uses custom or non-Python base image "pytorchlab/pytorch:2.4.1-cpu-py3.11-slim" in a Python environment. This may result in incorrect dependencies and/or incorrect behavior. Consider using the "DockerRunner" to run this task in a container.
  warnings.warn(


    
    [notice] A new release of pip is available: 26.0.1 -> 26.2.1
    [notice] To update, run: pip install --upgrade pip
    [KFP Executor 2026-08-27 09:12:25,767 INFO]: Looking for component `train_model` in --component_module_path `/tmp/tmp.1Y5B3YfVxB/ephemeral_component.py`
    [KFP Executor 2026-08-27 09:12:25,767 INFO]: Loading KFP component "train_model" from /tmp/tmp.1Y5B3YfVxB/ephemeral_component.py (directory "/tmp/tmp.1Y5B3YfVxB" and module name "ephemeral_component")
    [KFP Executor 2026-08-27 09:12:25,768 INFO]: Got executor_input:
    {
        "inputs": {
            "artifacts": {
                "input_dataset": {
                    "artifacts": [
                        {
                            "name": "output_dataset",
                            "type": {
                                "schemaTitle": "system.Dataset",
                                "schemaVersion": "0.0.1"
                            },
                            "uri": "/home/ridwanfa

In [17]:
PROJECT_ID = os.environ["GCP_PROJECT_ID"]
PIPELINE_ROOT = "gs://my-model-training/pipeline_root/kfp"

In [18]:
job = aiplatform.PipelineJob(
    display_name="pipeline",
    template_path="pipeline.yaml",
    pipeline_root=PIPELINE_ROOT,
    parameter_values={
        "source": "gs://my-model-training/datasets/dataset.csv",
    },
)

In [19]:
SERVICE_ACCOUNT = os.environ["GCP_SERVICE_ACCOUNT"]
job.submit(service_account = SERVICE_ACCOUNT)

Creating PipelineJob
PipelineJob created. Resource name: projects/344969539300/locations/us-central1/pipelineJobs/pipeline-20260827091238
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/344969539300/locations/us-central1/pipelineJobs/pipeline-20260827091238')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/pipeline-20260827091238?project=344969539300
